# Geographic categories: hierarchy, codes and coverage

This focused notebook completes the initial **geographic categorical** review of
`basin`, `subvillage`, `region`, `region_code`, `district_code`, `lga`, `ward`. It describes the supplied training and test predictors,
then uses the labelled training rows to identify relationships worth
carrying into a leakage-safe modelling pipeline.

Coordinates have their own paired audit; this notebook examines the named and encoded geographic context.

This is exploratory evidence, not fitted preprocessing. Category pooling,
imputation, encoding and scaling must be learned inside each training fold.


## Consistent audit contract

Every focused predictor audit answers the same questions before adding
type-specific checks:

1. What is explicitly missing, and what looks like a sentinel?
2. What range or category coverage is present in training and test?
3. How much of the test set is exposed to unseen training levels?
4. Does the labelled distribution vary enough to justify retaining the field?
5. What exact baseline treatment follows from the evidence?

Target-rate tables flag support rather than treating tiny groups as reliable.
Train/test comparisons are descriptive and do not use the hidden test labels.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

source_directory = str(Path("../src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    MISSING_CATEGORY,
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    cramer_v,
    hierarchy_conflicts,
    hierarchy_summary,
    numeric_summary,
    numeric_target_summary,
    normalise_categories,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = Path("../data")
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)
audited_features = ['basin', 'subvillage', 'region', 'region_code', 'district_code', 'lga', 'ward']
assert set(audited_features).issubset(training_features.columns)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {len(audited_features)} predictors."
)


Validated 59,400 training rows and 14,850 test rows for 7 predictors.


## 1. Missingness, cardinality and test coverage

Source blanks, pandas nulls and configured sentinel strings are reported separately.
Semantic sentinels such as `unknown` stay visible in frequency and target tables;
they are not silently merged with blank values.
Rare means fewer than 50 training rows; it is a diagnostic threshold, not
a preprocessing choice. Total-variation distance compares marginal shares.


In [2]:
category_overview = categorical_summary(
    training_features,
    test_features,
    audited_features,
    rare_threshold=50,
    sentinel_tokens_by_column={},
)
display(category_overview)


,training explicit missing,training source blank rows,training sentinel rows,test explicit missing,test source blank rows,test sentinel rows,training levels,test levels,training levels with <50 rows,training rows in rare levels (%),test-only levels,test rows in unseen levels (%),training-only levels,marginal total-variation distance
feature,,,,,,,,,,,,,,
basin,0,0,0,0,0,0,9,9,0,0.00,0,0.00,0,0.0123
subvillage,0,371,0,0,99,0,19287,8443,19227,87.32,2138,16.09,12982,0.4734
region,0,0,0,0,0,0,21,21,0,0.00,0,0.00,0,0.0190
region_code,0,0,0,0,0,0,27,26,1,0.00,0,0.00,1,0.0195
district_code,0,0,0,0,0,0,20,20,3,0.07,0,0.00,0,0.0101
lga,0,0,0,0,0,0,125,125,2,0.04,0,0.00,0,0.0359
ward,0,0,0,0,0,0,2092,1959,1771,57.35,6,0.07,139,0.1542


## 2. Most common values


In [3]:
for feature in audited_features:
    print()
    print(feature)
    display(category_frequency_table(training_features, test_features, feature, top_n=10))



basin


,training rows,training (%),test rows,test (%)
basin,,,,
lake victoria,10248,17.25,2623,17.66
pangani,8940,15.05,2203,14.84
rufiji,7976,13.43,2011,13.54
internal,7785,13.11,1857,12.51
lake tanganyika,6432,10.83,1620,10.91
wami / ruvu,5987,10.08,1590,10.71
lake nyasa,5085,8.56,1247,8.4
ruvuma / southern coast,4493,7.56,1094,7.37
lake rukwa,2454,4.13,605,4.07



subvillage


,training rows,training (%),test rows,test (%)
subvillage,,,,
madukani,508,0.86,121,0.81
shuleni,506,0.85,140,0.94
majengo,502,0.85,129,0.87
kati,373,0.63,94,0.63
<missing/blank>,371,0.62,99,0.67
mtakuja,262,0.44,60,0.4
sokoni,232,0.39,62,0.42
m,187,0.31,56,0.38
muungano,172,0.29,43,0.29



region


...

,training rows,training (%),test rows,test (%)
district_code,,,,
1,12203,20.54,3096,20.85
2,11173,18.81,2756,18.56
3,9998,16.83,2523,16.99
4,8999,15.15,2254,15.18
5,4356,7.33,1072,7.22
6,4074,6.86,1034,6.96
7,3343,5.63,823,5.54
8,1043,1.76,239,1.61
30,995,1.68,261,1.76



lga


,training rows,training (%),test rows,test (%)
lga,,,,
njombe,2503,4.21,625,4.21
arusha rural,1252,2.11,269,1.81
moshi rural,1251,2.11,315,2.12
bariadi,1177,1.98,308,2.07
rungwe,1106,1.86,275,1.85
kilosa,1094,1.84,274,1.85
kasulu,1047,1.76,275,1.85
mbozi,1034,1.74,252,1.7
meru,1009,1.7,235,1.58



ward


,training rows,training (%),test rows,test (%)
ward,,,,
igosi,307,0.52,79,0.53
imalinyi,252,0.42,66,0.44
siha kati,232,0.39,65,0.44
mdandu,231,0.39,61,0.41
nduruma,217,0.37,44,0.3
kitunda,203,0.34,57,0.38
mishamo,203,0.34,48,0.32
msindo,201,0.34,42,0.28
chalinze,196,0.33,42,0.28


## 3. Relationship with `status_group`

The tables display the most supported levels first and mark whether each
level has at least 200 training rows. Small groups are leads
for later validation, not stable target encodings.


In [4]:
for feature in audited_features:
    print()
    print(feature)
    profile = categorical_target_profile(
        training_data,
        feature,
        minimum_support=200,
    )
    display(profile.head(15))



basin


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
basin,,,,,
lake victoria,10248,True,49.77,9.65,40.58
pangani,8940,True,60.09,5.34,34.57
rufiji,7976,True,63.54,5.48,30.98
internal,7785,True,57.57,7.15,35.27
lake tanganyika,6432,True,48.31,11.54,40.16
wami / ruvu,5987,True,52.38,4.49,43.13
lake nyasa,5085,True,65.37,4.92,29.71
ruvuma / southern coast,4493,True,37.17,7.26,55.58
lake rukwa,2454,True,40.75,11.00,48.25



subvillage


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
subvillage,,,,,
madukani,508,True,48.82,8.46,42.72
shuleni,506,True,45.65,8.70,45.65
majengo,502,True,46.61,7.17,46.22
kati,373,True,57.64,11.53,30.83
<missing/blank>,371,True,55.26,0.27,44.47
mtakuja,262,True,54.20,5.34,40.46
sokoni,232,True,47.84,5.60,46.55
m,187,False,66.31,1.60,32.09
muungano,172,False,45.93,7.56,46.51



region


...

status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
district_code,,,,,
1,12203,True,53.74,10.55,35.71
2,11173,True,55.52,7.55,36.93
3,9998,True,49.55,6.99,43.46
4,8999,True,62.17,5.66,32.17
5,4356,True,56.91,4.45,38.64
6,4074,True,50.44,5.69,43.86
7,3343,True,60.25,6.40,33.35
8,1043,True,56.47,5.08,38.45
30,995,True,69.25,8.64,22.11



lga


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
lga,,,,,
njombe,2503,True,80.18,3.76,16.06
arusha rural,1252,True,69.89,3.83,26.28
moshi rural,1251,True,58.59,9.51,31.89
bariadi,1177,True,49.28,34.75,15.97
rungwe,1106,True,61.12,14.56,24.32
kilosa,1094,True,53.66,6.67,39.67
kasulu,1047,True,58.36,19.20,22.45
mbozi,1034,True,43.52,6.77,49.71
meru,1009,True,65.11,3.17,31.71



ward


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
ward,,,,,
igosi,307,True,94.14,0.00,5.86
imalinyi,252,True,95.24,1.19,3.57
siha kati,232,True,98.28,0.86,0.86
mdandu,231,True,87.45,5.63,6.93
nduruma,217,True,63.13,7.37,29.49
kitunda,203,True,79.31,0.00,20.69
mishamo,203,True,21.18,7.39,71.43
msindo,201,True,69.15,6.97,23.88
chalinze,196,False,78.06,0.00,21.94


## 4. Related-field consistency

A deterministic child-to-parent mapping makes the parent derivable from
the child in this dataset. That is redundancy evidence, not automatic
permission to discard the child: granularity, unseen levels and model
behaviour still determine which representation is safer.


In [5]:
hierarchy_relationships = [('region', 'region_code'), ('lga', 'region'), ('ward', 'lga'), ('subvillage', 'ward')]
display(
    hierarchy_summary(
        training_features,
        test_features,
        hierarchy_relationships,
    )
)
for child, parent in hierarchy_relationships:
    conflicts = hierarchy_conflicts(training_features, child, parent)
    if not conflicts.empty:
        print()
        print(f"Training conflicts for {child} -> {parent}")
        display(conflicts)


complete rows  child levels  parent levels  \
relationship          frame                                                  
region -> region_code training          59400            21             27   
                      test              14850            21             26   
lga -> region         training          59400           125             21   
                      test              14850           125             21   
ward -> lga           training          59400          2092            125   
                      test              14850          1959            125   
subvillage -> ward    training          59029         19287           2080   
                      test              14751          8443           1946   

                                ambiguous child levels  \
relationship          frame                              
region -> region_code training                       7   
                      test                           7   
lga -> region         training                       0   
                      test                           0   
ward -> lga           training                      88   
                      test                          77   
subvillage -> ward    training                    2450   
                      test                         831   

                                rows in ambiguous child levels  \
relationship          frame                                      
region -> region_code training                           19892   
                      test                                4990   
lga -> region         training                               0   
                      test                                   0   
ward -> lga           training                            5414   
                      test                                1267   
subvillage -> ward    training                           23918   
                      test                                4661   

                                deterministic child-to-parent  \
relationship          frame                                     
region -> region_code training                          False   
                      test                              False   
lga -> region         training                           True   
                      test                               True   
ward -> lga           training                          False   
                      test                              False   
subvillage -> ward    training                          False   
                      test                              False   

                                one-to-one level mapping  
relationship          frame                               
region -> region_code training                     False  
                      test                         False  
lga -> region         training                     False  
                      test                         False  
ward -> lga           training                     False  
                      test                         False  
subvillage -> ward    training                     False  
                      test                         False


Training conflicts for region -> region_code


,rows,parent levels,parents
child,,,
shinyanga,4982,3,"11, 14, 17"
arusha,3350,2,"2, 24"
mwanza,3102,2,"17, 19"
pwani,2635,3,"40, 6, 60"
tanga,2547,2,"4, 5"
mtwara,1730,3,"9, 90, 99"
lindi,1546,3,"18, 8, 80"



Training conflicts for ward -> lga


,rows,parent levels,parents
child,,,
nduruma,217,2,"arusha rural, ukerewe"
kitunda,203,2,"ilala, sikonge"
msindo,201,2,"namtumbo, same"
chanika,171,2,"handeni, ilala"
mtwango,153,2,"mufindi, njombe"
itete,137,2,"rungwe, ulanga"
mlangali,125,2,"ludewa, mbozi"
nkoma,122,2,"bariadi, meatu"
mahongole,121,2,"mbarali, njombe"



Training conflicts for subvillage -> ward


,rows,parent levels,parents
child,,,
madukani,508,159,"badugu, buchambi, bukundi, bumbuli, busi, busisi, butimba, chalinze, chandam..."
shuleni,506,191,"barray, berega, bubiki, bukundi, bumbuta, bumera, bunda, busagara, busolwa, ..."
majengo,502,216,"bendera, berega, bonde la songwe, bugarama, bukene, bunambiu, bunda, busi, b..."
kati,373,84,"barray, berege, bumera, bungu, busi, busolwa, butimba, bwisya, chanika, daba..."
<missing/blank>,371,21,"bukanda, bukiko, bukindo, bwiro, bwisya, chamkoroma, hogoro, iduo, kibaigwa,..."
mtakuja,262,116,"bombambili, bugarama, bukoli, bukura, bulungwa, bumbuta, butimba, buziku, bw..."
sokoni,232,82,"berega, buziku, chakwale, chawi, chemchem, chingungwe, chitekete, chiugutwa,..."
m,187,20,"bumilayinga, idunda, igombavanu, igowole, ihalimba, ihowanza, isalavanu, ita..."
muungano,172,75,"bonde la songwe, bungu, chanzuru, chihanga, chikongola, chilionwa, chimala, ..."


## 5. Geographic code semantics

`region_code` and `district_code` are labels despite their integer dtype.
District codes are checked both alone and as a region/district composite.


In [6]:
training_geo = training_features.copy()
test_geo = test_features.copy()
for frame in (training_geo, test_geo):
    frame["region_district"] = (
        frame["region_code"].astype("string")
        + ":"
        + frame["district_code"].astype("string")
    )
composite_summary = categorical_summary(
    training_geo,
    test_geo,
    ["region_district"],
)
display(composite_summary)

lga_mapping = hierarchy_summary(
    training_geo,
    test_geo,
    [("lga", "region_district")],
)
display(lga_mapping)


,training explicit missing,training source blank rows,training sentinel rows,test explicit missing,test source blank rows,test sentinel rows,training levels,test levels,training levels with <20 rows,training rows in rare levels (%),test-only levels,test rows in unseen levels (%),training-only levels,marginal total-variation distance
feature,,,,,,,,,,,,,,
region_district,0,0,0,0,0,0,130,129,3,0.03,0,0.0,1,0.0369


complete rows  child levels  parent levels  \
relationship           frame                                                  
lga -> region_district training          59400           125            130   
                       test              14850           125            129   

                                 ambiguous child levels  \
relationship           frame                              
lga -> region_district training                      37   
                       test                          35   

                                 rows in ambiguous child levels  \
relationship           frame                                      
lga -> region_district training                           21174   
                       test                                5057   

                                 deterministic child-to-parent  \
relationship           frame                                     
lga -> region_district training                          False   
                       test                              False   

                                 one-to-one level mapping  
relationship           frame                               
lga -> region_district training                     False  
                       test                         False

## Training/test handoff


In [7]:
display(
    category_overview[[
        "training levels",
        "test levels",
        "test-only levels",
        "test rows in unseen levels (%)",
        "training rows in rare levels (%)",
        "marginal total-variation distance",
    ]].sort_values("test rows in unseen levels (%)", ascending=False)
)


,training levels,test levels,test-only levels,test rows in unseen levels (%),training rows in rare levels (%),marginal total-variation distance
feature,,,,,,
subvillage,19287,8443,2138,16.09,87.32,0.4734
ward,2092,1959,6,0.07,57.35,0.1542
basin,9,9,0,0.00,0.00,0.0123
region,21,21,0,0.00,0.00,0.0190
region_code,27,26,0,0.00,0.00,0.0195
district_code,20,20,0,0.00,0.07,0.0101
lga,125,125,0,0.00,0.04,0.0359


## Decision register

The register separates observed evidence from the proposed baseline action.
A retained field is still a candidate: later validation must show whether it
improves generalisation and whether a coarser related representation is safer.


In [8]:
decision_register = pd.DataFrame([{'feature': 'basin', 'quality finding': 'Nine complete levels; train/test total-variation distance is 1.23%.', 'baseline treatment': 'Retain as a stable low-cardinality category.', 'risk to verify': 'Hydrological basin crosses administrative geography.'}, {'feature': 'subvillage', 'quality finding': '19,287 levels; 87.32% of rows are in <50 groups and 16.09% of test rows are unseen.', 'baseline treatment': 'Exclude from first one-hot baseline; validate hashing/frequency treatment separately.', 'risk to verify': 'Extreme sparsity and reused names encourage memorisation.'}, {'feature': 'region', 'quality finding': 'Twenty-one complete levels; functional rates range from 29.75% to 78.22%.', 'baseline treatment': 'Retain as an interpretable low-cardinality back-off.', 'risk to verify': 'Random validation may reward geographic memorisation.'}, {'feature': 'region_code', 'quality finding': 'Twenty-seven categorical codes; region and code are not simple duplicates.', 'baseline treatment': 'Cast to category and compare with named region.', 'risk to verify': 'Do not scale or interpret code distance.'}, {'feature': 'district_code', 'quality finding': 'Twenty reused numeric labels; code zero is not universally missing.', 'baseline treatment': 'Cast to category and combine with region_code if used.', 'risk to verify': 'The same code can occur in different regions.'}, {'feature': 'lga', 'quality finding': '125 complete levels, no unseen test levels, and deterministic mapping to region.', 'baseline treatment': 'Retain; compare its signal against region back-off.', 'risk to verify': 'Add an LGA/region-grouped validation sensitivity check.'}, {'feature': 'ward', 'quality finding': '2,092 levels; raw unseen exposure is 0.07%, or 0.08% for LGA+ward.', 'baseline treatment': 'Retain via LGA+ward with fold-fitted rare/unseen handling.', 'risk to verify': 'Shared ward names need the LGA context.'}])
display(decision_register.set_index("feature"))


,quality finding,baseline treatment,risk to verify
feature,,,
basin,Nine complete levels; train/test total-variation distance is 1.23%.,Retain as a stable low-cardinality category.,Hydrological basin crosses administrative geography.
subvillage,"19,287 levels; 87.32% of rows are in <50 groups and 16.09% of test rows are ...",Exclude from first one-hot baseline; validate hashing/frequency treatment se...,Extreme sparsity and reused names encourage memorisation.
region,Twenty-one complete levels; functional rates range from 29.75% to 78.22%.,Retain as an interpretable low-cardinality back-off.,Random validation may reward geographic memorisation.
region_code,Twenty-seven categorical codes; region and code are not simple duplicates.,Cast to category and compare with named region.,Do not scale or interpret code distance.
district_code,Twenty reused numeric labels; code zero is not universally missing.,Cast to category and combine with region_code if used.,The same code can occur in different regions.
lga,"125 complete levels, no unseen test levels, and deterministic mapping to reg...",Retain; compare its signal against region back-off.,Add an LGA/region-grouped validation sensitivity check.
ward,"2,092 levels; raw unseen exposure is 0.07%, or 0.08% for LGA+ward.",Retain via LGA+ward with fold-fitted rare/unseen handling.,Shared ward names need the LGA context.


### Handoff to modelling

Treat administrative codes as unordered categories and compare nested levels by validation rather than retaining every hierarchy level by default.

- Preserve raw source frames and implement the stated sentinel rules on copies.
- Fit imputers, rare-level grouping and encoders on each training fold only.
- Map unseen validation or test categories to an explicit fallback.
- Compare the stated baseline treatment with a simple omission ablation.
- Revisit target-rate observations after the reproducible stratified split exists.
